<a href="https://colab.research.google.com/github/ianwambua-bit/Machine-Learning_Ian-Wambua/blob/main/Financial%20Inclusion%20Classification%20(Rwanda%20Demographic%20Survey).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course Project: Predicting Financial Inclusion in Africa (Bank Account Ownership)

## 1. Project Problem Statement & Intended AI/ML Outcome

### Problem Restatement
Access to formal banking infrastructure remains limited across rural and low-income demographics in East Africa. Without formal financial inclusion, households face barriers in accessing credit, insurance, and formal savings mechanisms. Identifying the demographic, geographic, and socioeconomic determinants of bank account ownership allows policymakers and microfinance institutions to design targeted mobile banking and financial literacy interventions.

### Intended AI/ML Outcome
Develop a supervised binary classification model that accurately predicts whether a surveyed individual maintains an active bank account (`bank_account` = 1 vs. 0) based on household and individual demographic attributes.

## 2. Data Requirements

The solution requires individual-level demographic and economic survey data, containing:
1. Geographic location indicators (Urban vs. Rural).
2. Infrastructure access flags (Access to cellphone).
3. Household metrics (Household size, relationship to head of household).
4. Individual demographics (Age, gender, marital status, education level attained).
5. Employment classifications (Farming, informal employment, self-employed, remittance-dependent).

In [2]:
# ==============================================================================
# 3. Dataset Ingestion & Inspection (Google Colab Code Cell)
# ==============================================================================

import pandas as pd
import numpy as np

# Load the provided Rwanda Financial Inclusion dataset schema (RwandaFn.pdf)
data_url = "https://drive.google.com/file/d/1P9Xot60xMeekWs_wj2ua8J1Vko1rH7UJ/view?usp=drive_link"

# Example local/sample dataframe representing the exact Rwanda survey schema:
df = pd.DataFrame({
    'country': ['Rwanda', 'Rwanda', 'Rwanda', 'Rwanda', 'Rwanda'],
    'year': [2016, 2016, 2016, 2016, 2016],
    'uniqueid': ['uniqueid_1', 'uniqueid_2', 'uniqueid_3', 'uniqueid_4', 'uniqueid_5'],
    'location_type': ['Urban', 'Urban', 'Rural', 'Urban', 'Rural'],
    'cellphone_access': ['Yes', 'Yes', 'No', 'Yes', 'No'],
    'household_size': [7, 4, 6, 1, 4],
    'age_of_respondent': [18, 26, 42, 41, 17],
    'gender_of_respondent': ['Female', 'Female', 'Male', 'Male', 'Female'],
    'relationship_with_head': ['Child', 'Spouse', 'Head of Household', 'Head of Household', 'Child'],
    'marital_status': ['Single/Never Married', 'Married/Living together', 'Married/Living together', 'Single/Never Married', 'Single/Never Married'],
    'education_level': ['Secondary education', 'Secondary education', 'No formal education', 'Tertiary education', 'Primary education'],
    'job_type': ['Remittance Dependent', 'Self employed', 'Farming and Fishing', 'Informally employed', 'Informally employed'],
    'bank_account': ['No', 'Yes', 'No', 'Yes', 'No']
})

print("Financial Inclusion Survey Dataset Structure:")
display(df.head())
print("\nTarget Variable Distribution:")
print(df['bank_account'].value_counts())

Financial Inclusion Survey Dataset Structure:


,country,year,uniqueid,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type,bank_account
0,Rwanda,2016,uniqueid_1,Urban,Yes,7,18,Female,Child,Single/Never Married,Secondary education,Remittance Dependent,No
1,Rwanda,2016,uniqueid_2,Urban,Yes,4,26,Female,Spouse,Married/Living together,Secondary education,Self employed,Yes
2,Rwanda,2016,uniqueid_3,Rural,No,6,42,Male,Head of Household,Married/Living together,No formal education,Farming and Fishing,No
3,Rwanda,2016,uniqueid_4,Urban,Yes,1,41,Male,Head of Household,Single/Never Married,Tertiary education,Informally employed,Yes
4,Rwanda,2016,uniqueid_5,Rural,No,4,17,Female,Child,Single/Never Married,Primary education,Informally employed,No



Target Variable Distribution:
bank_account
No     3
Yes    2
Name: count, dtype: int64


## 4. Potential Inputs (Features) and Output (Target)

### Input Features ($X$)
* `location_type` (Binary Categorical): Urban or Rural residency.
* `cellphone_access` (Binary Categorical): Whether the respondent has mobile phone access.
* `household_size` (Numerical): Total individuals residing in the household.
* `age_of_respondent` (Numerical): Age of the interviewee in years.
* `gender_of_respondent` (Categorical): Male or Female.
* `relationship_with_head` (Categorical): Head of Household, Spouse, Child, Parent, Other relative.
* `marital_status` (Categorical): Married/Living together, Single, Widowed, Divorced/Separated.
* `education_level` (Ordinal Categorical): No formal education, Primary, Secondary, Vocational, Tertiary.
* `job_type` (Categorical): Occupational status (Farming & Fishing, Self-employed, Formally employed, Remittance Dependent, etc.).

### Target Output ($y$)
* `bank_account` (Binary Categorical / Label-Encoded integer: $1 = \text{Yes}$, $0 = \text{No}$): Indicates formal bank account ownership.



## 5. Proposed Machine Learning Workflow

```text
+----------------------------------------------------------------------------------------------------+
| 1. DATA EXTRACTION & INGESTION                                                                     |
|    - Tabular survey data ingestion (Rwanda / East Africa Financial Inclusion)                      |
|    - Identifier removal ('uniqueid', 'country', 'year')                                            |
+-------------------------------------------------+--------------------------------------------------+
                                                  |
                                                  v
+----------------------------------------------------------------------------------------------------+
| 2. EXPLORATORY ANALYSIS & PREPROCESSING                                                            |
|    - Class imbalance assessment (Class 0: No >> Class 1: Yes)                                      |
|    - One-Hot Encoding for nominal features (job_type, marital_status)                              |
|    - Ordinal encoding for education_level; MinMax scaling for age and household_size               |
|    - Train/Validation/Test stratified split                                                        |
+-------------------------------------------------+--------------------------------------------------+
                                                  |
                                                  v
+----------------------------------------------------------------------------------------------------+
| 3. BASELINE MODEL (Tip 2: Start with the Simplest Model)                                           |
|    - Dummy classifier / Logistic Regression to validate pipeline and set baseline metric          |
|    - Evaluate ROC-AUC, F1-Score, and Precision-Recall rather than raw accuracy                     |
+-------------------------------------------------+--------------------------------------------------+
                                                  |
                                                  v
+----------------------------------------------------------------------------------------------------+
| 4. MODEL SELECTION & TUNING (Tips 3 & 4: Controlled Tuning & Learning Curves)                      |
|    - Train Tree-based models (Random Forest, XGBoost Classifier) with SMOTE / class weighting      |
|    - Compare models under matched experiment iterations (100 runs vs. 100 runs)                    |
|    - Track validation learning curves across dataset subsets to detect overfitting/saturation      |
+-------------------------------------------------+--------------------------------------------------+
                                                  |
                                                  v
+----------------------------------------------------------------------------------------------------+
| 5. EVALUATION OF TRADE-OFFS & ERROR ANALYSIS (Tips 5 & 6)                                          |
|    - Analyze False Positives vs. False Negatives (Cost of credit denial vs. default risk)          |
|    - Validate conditional independence and feature attribution (SHAP values)                      |
|    - Export model artifacts for inference                                                          |
+----------------------------------------------------------------------------------------------------+
```

### Workflow Component Explanation
1. **Ingestion & Data Scrubbing**: Loads the survey data and drops invariant administrative fields (`country`, `year`, `uniqueid`).
2. **Preprocessing**: Converts target labels to binary integers, applies dummy encoding to nominal categories, and standardizes continuous distributions.
3. **Simple Baseline Establishment**: Trains an interpretable linear classifier as a benchmark before testing complex ensembles.
4. **Controlled Model Comparison**: Evaluates ensemble methods (XGBoost, Random Forest) using identical cross-validation runs to eliminate hyperparameter optimization bias.
5. **Trade-off Evaluation**: Weighs precision against recall to minimize class imbalance bias, ensuring predictions capture marginalized positive cases (`bank_account = 1`).